In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install -q dagshub mlflow

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.7/49.7 kB 3.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.5/50.5 kB 4.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 273.1/273.1 kB 15.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 116.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 124.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 87.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 264.7/264.7 kB 21.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 132.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.2/68.2 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.8/147.8 kB 16.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 12.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2

In [ ]:
!pip install kaggle

In [ ]:
!mkdir -p ~/.kaggle
!cp /content/drive/MyDrive/cs231n/assignments/4/kaggle.json ~/.kaggle/kaggle.json
! chmod 600 ~/.kaggle/kaggle.json

In [ ]:
!kaggle competitions download -c walmart-recruiting-store-sales-forecasting
!unzip -q walmart-recruiting-store-sales-forecasting.zip

100% 2.70M/2.70M [00:00<00:00, 26.5MB/s]



In [ ]:
!unzip -q train.csv.zip
!unzip -q stores.csv.zip
!unzip -q test.csv.zip
!unzip -q features.csv.zip

unzip:  cannot find or open stores.csv.zip, stores.csv.zip.zip or stores.csv.zip.ZIP.


In [ ]:
import mlflow
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import mean_squared_error, mean_absolute_error

from lightgbm import LGBMRegressor

In [ ]:
import dagshub
import mlflow

dagshub.init(repo_owner='tsarc21', repo_name='Walmart-Recruiting---Store-Sales-Forecasting', mlflow=True)


❗❗❗ AUTHORIZATION REQUIRED ❗❗❗

Output()



Open the following link in your browser to authorize the client:
https://dagshub.com/login/oauth/authorize?state=bc0c2d68-7e68-4768-a51e-eee3d8ae1e0f&client_id=32b60ba385aa7cecf24046d8195a71c07dd345d9657977863b52e7748e0f0f28&middleman_request_id=f76dde25a0c5cddd8c4e7af64ae71ea58a58fb31762b5271333095e66d12e6b1




Accessing as tsarc21

Initialized MLflow to track repo "tsarc21/Walmart-Recruiting---Store-Sales-Forecasting"

Repository tsarc21/Walmart-Recruiting---Store-Sales-Forecasting initialized!

In [ ]:
import pandas as pd

train = pd.read_csv("train.csv")
stores = pd.read_csv("stores.csv")
features = pd.read_csv("features.csv")

train["Date"] = pd.to_datetime(train["Date"])
features["Date"] = pd.to_datetime(features["Date"])

df = train.merge(stores, on="Store", how="left")

df = df.merge(
    features,
    on=["Store", "Date", "IsHoliday"],
    how="left"
)

df = df.sort_values(["Store", "Dept", "Date"]).reset_index(drop=True)


df["Year"] = df["Date"].dt.year
df["Month"] = df["Date"].dt.month
df["Week"] = df["Date"].dt.isocalendar().week.astype(int)
df["Day"] = df["Date"].dt.day
df["DayOfWeek"] = df["Date"].dt.dayofweek
df["Quarter"] = df["Date"].dt.quarter

df["IsHoliday"] = df["IsHoliday"].astype(int)


group = df.groupby(["Store", "Dept"])

df["Lag_1"] = group["Weekly_Sales"].shift(1)
df["Lag_2"] = group["Weekly_Sales"].shift(2)
df["Lag_52"] = (
    df.groupby(["Store","Dept"])["Weekly_Sales"]
    .shift(52)
)
df["Rolling_4"] = (
    group["Weekly_Sales"]
    .transform(lambda x: x.shift(1).rolling(4, min_periods=1).mean())
)
df["Rolling_12"] = (
    df.groupby(["Store","Dept"])["Weekly_Sales"]
    .transform(
        lambda x: x.shift(1).rolling(12).mean()
    )
)
df = df.dropna().reset_index(drop=True)

split_1 = df["Date"].max() - pd.Timedelta(days=90)
split_2 = df["Date"].max() - pd.Timedelta(days=45)

train_df = df[df["Date"] < split_1].copy()

val_df = df[
    (df["Date"] >= split_1) &
    (df["Date"] < split_2)
].copy()

test_df = df[df["Date"] >= split_2].copy()


target = "Weekly_Sales"

X_train = train_df.drop(columns=[target, "Date"])
y_train = train_df[target]

X_val = val_df.drop(columns=[target, "Date"])
y_val = val_df[target]

X_test = test_df.drop(columns=[target, "Date"])
y_test = test_df[target]

In [ ]:
import numpy as np
import joblib

from lightgbm import LGBMRegressor

from sklearn.compose import ColumnTransformer

from sklearn.pipeline import Pipeline

from sklearn.preprocessing import OneHotEncoder

from sklearn.impute import SimpleImputer

from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error
)

from sklearn.model_selection import RandomizedSearchCV

import mlflow
import mlflow.sklearn

categorical_features = [
    "Store",
    "Dept",
    "Type"
]

numeric_features = [
    c for c in X_train.columns
    if c not in categorical_features
]

numeric_transformer = Pipeline([
    (
        "imputer",
        SimpleImputer(strategy="median")
    )
])

categorical_transformer = Pipeline([
    (
        "imputer",
        SimpleImputer(strategy="most_frequent")
    ),
    (
        "encoder",
        OneHotEncoder(handle_unknown="ignore")
    )
])

preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            numeric_transformer,
            numeric_features
        ),
        (
            "cat",
            categorical_transformer,
            categorical_features
        )
    ]
)


model = LGBMRegressor(
    objective="regression",
    random_state=42,

    n_estimators=1000,
    learning_rate=0.03,

    num_leaves=31,
    max_depth=6,

    min_child_samples=100,

    subsample=0.8,
    colsample_bytree=0.8,

    reg_alpha=1,
    reg_lambda=5,

    verbosity=-1
)

pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", model)
])

param_dist = {

    "model__n_estimators":
        [500,1000,1500,2000],

    "model__learning_rate":
        [0.01,0.03,0.05,0.1],

    "model__num_leaves":
        [16,31,64],

    "model__max_depth":
        [5,6,8,-1],

    "model__min_child_samples":
        [50,100,200],

    "model__subsample":
        [0.7,0.8,0.9,1.0],

    "model__colsample_bytree":
        [0.7,0.8,0.9,1.0],

    "model__reg_alpha":
        [0,0.1,0.5,1],

    "model__reg_lambda":
        [0.5,1,2,5]
}


search = RandomizedSearchCV(

    estimator=pipeline,

    param_distributions=param_dist,

    n_iter=50,

    scoring="neg_root_mean_squared_error",

    cv=3,

    random_state=42,

    verbose=2,

    n_jobs=-1

)

mlflow.set_experiment("LGBM_Experiment")
with mlflow.start_run(run_name="LGBM_Final_model"):

    search.fit(X_train, y_train)

    best_model = search.best_estimator_


    train_pred = best_model.predict(X_train)
    val_pred = best_model.predict(X_val)
    test_pred = best_model.predict(X_test)


    train_rmse = np.sqrt(mean_squared_error(y_train, train_pred))
    train_mae = mean_absolute_error(y_train, train_pred)

    val_rmse = np.sqrt(mean_squared_error(y_val, val_pred))
    val_mae = mean_absolute_error(y_val, val_pred)

    test_rmse = np.sqrt(mean_squared_error(y_test, test_pred))
    test_mae = mean_absolute_error(y_test, test_pred)

    print("Best Parameters:")
    print(search.best_params_)

    print(f"TRAIN      RMSE: {train_rmse:.4f}  MAE: {train_mae:.4f}")
    print(f"VALIDATION RMSE: {val_rmse:.4f}  MAE: {val_mae:.4f}")


    mlflow.log_params(search.best_params_)


    mlflow.log_metric("train_rmse", train_rmse)
    mlflow.log_metric("train_mae", train_mae)

    mlflow.log_metric("val_rmse", val_rmse)
    mlflow.log_metric("val_mae", val_mae)


    mlflow.sklearn.log_model(
        sk_model=best_model,
        name="lightgbm_pipeline",
        serialization_format="cloudpickle"
    )


    joblib.dump(best_model, "best_lightgbm_pipeline.pkl")


Fitting 3 folds for each of 50 candidates, totalling 150 fits


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


Best Parameters:
{'model__subsample': 0.7, 'model__reg_lambda': 5, 'model__reg_alpha': 0.1, 'model__num_leaves': 16, 'model__n_estimators': 2000, 'model__min_child_samples': 50, 'model__max_depth': 6, 'model__learning_rate': 0.05, 'model__colsample_bytree': 1.0}
TRAIN      RMSE: 2442.6168  MAE: 1279.0462
VALIDATION RMSE: 2902.3785  MAE: 1451.6190


2026/07/09 11:35:17 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run LGBM_Final_model at: https://dagshub.com/tsarc21/Walmart-Recruiting---Store-Sales-Forecasting.mlflow/#/experiments/3/runs/d704b5ccdf4e409e8bca244d7fc94e01
🧪 View experiment at: https://dagshub.com/tsarc21/Walmart-Recruiting---Store-Sales-Forecasting.mlflow/#/experiments/3
